# Very guided Qwen2.5-7B LoRA training

This **is the training notebook**. It is intentionally slow to read and easy to
follow. The cells are small because the goal is not merely to finish a run: the
goal is for you to understand what the run is doing.

**ELI5:** We start with a large tutor that already knows language. LoRA adds a
small set of adjustable knobs. Training changes only those knobs; the original
model stays frozen.

Run one cell at a time. Read the explanation above a cell, run the cell, and look
at its output before continuing. If an assertion or gate fails, stop there. A
failure is useful information, not an invitation to skip the check.

This notebook uses Hugging Face **PEFT + TRL**. Microsoft's `loralib` repository
is background reading only and is not installed or copied into this project.

## Before opening this notebook: create the training environment

Use a dedicated Python 3.12 virtual environment. Do not use your system Python,
an application environment, or Python 3.14. Run these commands from the repository
root:

```bash
uv venv .venv-train --python 3.12
source .venv-train/bin/activate

uv pip install -r train/requirements.txt \
  --index-url https://download.pytorch.org/whl/cu128 \
  --extra-index-url https://pypi.org/simple

python -m ipykernel install --user \
  --name socratic-train \
  --display-name \"Python 3 (socratic training)\"

jupyter lab
```

In Jupyter, select the **Python 3 (socratic training)** kernel. `uv venv` creates
a normal virtual environment; Conda is not required. Run `nvidia-smi` first to
confirm that the NVIDIA driver can see the GPU.


## How to use this notebook

The notebook has three modes. You change only `RUN_MODE` in the setup section.

| Mode | What happens |
| --- | --- |
| `inspect` | CPU explanations, data checks, and the tokenizer/mask lesson. No model training. |
| `smoke` | The default. All checks, three base-model replies, and exactly one real GPU optimizer step. |
| `full` | The one fixed three-epoch training run. It also requires `CONFIRM_FULL_RUN = True`. |

**ELI5:** Smoke mode is like turning a car engine on for one second before a
long road trip. It checks that the engine, fuel, and dashboard work. It is not
the road trip, and it does not produce a pretend final adapter.

Before using this notebook, create the Python 3.12 CUDA environment described in
`train/README.md`. Do not start by clicking “Run All”; this lesson is designed to
be followed top-to-bottom with pauses.


## The fixed experiment, in plain English

We will train exactly one adapter for:

- model: `Qwen/Qwen2.5-7B-Instruct` at one immutable Hub revision;
- data: the committed 400-dialogue training pool;
- adapter: ordinary BF16 LoRA, not QLoRA;
- loss: only tokens written by the assistant count as answers to learn;
- duration: three epochs, one seed, no checkpoint competition;
- evaluation: the separate 48-case benchmark, not this notebook.

“Fixed” matters. We are trying to make one result explainable and reproducible,
not hunt through many recipes until one looks best.


## Four words to keep nearby

- **Base model:** the original Qwen weights before this experiment.
- **Adapter:** the small LoRA file containing the learned changes.
- **Token:** a small piece of text after the tokenizer splits a sentence.
- **Loss:** a number measuring how surprised the model was. Training tries to make
  this number smaller on the tokens we told it to learn.

The rest of the notebook spells out how each of those ideas appears in code.


## 1. Start with the smallest possible setup

First we load Python tools and check where the repository is. Nothing has been trained yet.


### Why this first code cell exists

`from __future__ import annotations` makes modern type annotations behave
consistently. `import sys` gives us information about the Python interpreter.
The `print` line is a visible receipt: it tells us which Python is running the
notebook.


In [ ]:
from __future__ import annotations

import sys

print("Python version:", sys.version.split()[0])


### Read the standard-library imports

These are tools that ship with Python. They do not train a model:

- `hashlib` computes file fingerprints;
- `importlib.metadata` reads installed package versions;
- `json` reads the JSONL data and writes logs;
- `math` checks whether a loss is finite;
- `os` sets a tokenizer environment switch;
- `random` and `subprocess` support reproducibility and repository checks;
- `tempfile` creates disposable smoke-run output;
- `time` measures duration;
- `Counter` counts data families;
- `Path` gives us safe file paths.


In [ ]:
import hashlib
import importlib.metadata
import json
import math
import os
import random
import subprocess
import tempfile
import time
from collections import Counter
from pathlib import Path


### Load PyTorch, with a helpful error

PyTorch is the engine that holds tensors, computes gradients, and talks to the
GPU. The `try` block attempts the import. If the package is absent, the `except`
block turns Python's cryptic import error into a setup instruction.


In [ ]:
try:
    import torch
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "PyTorch is missing. Create the dedicated environment using train/README.md."
    ) from exc


### Find the repository root

A notebook's current directory can vary depending on how Jupyter was launched.
This helper starts at the current directory and walks upward until it finds the
committed training file. That prevents us from accidentally reading data from a
different checkout.

- `start` is where the search begins.
- `start.parents` contains its parent directories.
- The `for` loop checks each candidate.
- The `raise` line stops with a useful message if this is not the Socratic repo.


In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data/train/dialogues.jsonl").exists():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the socratic repository.")


### Record the root and make project imports visible

`Path.cwd()` means “the directory where Jupyter is currently running.”
`find_repo_root` turns that into the repository root. Adding that root to
`sys.path` lets this notebook import the project's existing evaluation prompt
instead of copying a second version into the notebook.


In [ ]:
ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(ROOT))

print("Repository root:", ROOT)


### Pin the model identity

A model name can point to a moving label. The revision is the immutable commit
that makes this experiment refer to the same model files later.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MODEL_REVISION = "a09a35458c702b33eeacc393d103063234e8bc28"


In [ ]:
print("Model:", MODEL_ID)
print("Pinned revision:", MODEL_REVISION)


### Write down the LoRA recipe

These four values describe the size and placement of the adapter:

- `r` is the rank: how many small directions LoRA can learn;
- `alpha` scales the update;
- `dropout=0` means no random LoRA dropout in this fixed recipe;
- `all-linear` asks PEFT to cover every eligible linear projection.

There is no search loop around these values. Each is one declared experiment
choice.


In [ ]:
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.0
LORA_TARGET_MODULES = "all-linear"


### Write down the training values

`LEARNING_RATE` says how large each learned adjustment may be. `EPOCHS` says
how many times to read the pool. `MAX_LENGTH` caps the number of tokens in one
example so memory use is bounded.


In [ ]:
LEARNING_RATE = 2e-4
EPOCHS = 3
MAX_LENGTH = 1024


### Write down batch, loss, and precision choices

A micro-batch of one keeps the 24-GiB GPU within the intended memory budget.
Assistant-only loss is the important target rule: the model reads system and
user messages as context, but learns from assistant responses. `USE_QLORA=False`
records that this is ordinary BF16 LoRA, not 4-bit QLoRA.


In [ ]:
MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
ASSISTANT_ONLY_LOSS = True
USE_QLORA = False
DTYPE = torch.bfloat16


### Choose a mode, and make full training opt-in

The default is `smoke`, so a top-to-bottom run is limited to one real optimizer
step. `CONFIRM_FULL_RUN` is a second lock: changing the mode alone cannot start
the three-epoch job.


In [ ]:
SEED = 20260805
RUN_MODE = "smoke"  # Change to "inspect" or intentionally to "full".
CONFIRM_FULL_RUN = False


In [ ]:
allowed_modes = {"inspect", "smoke", "full"}

if RUN_MODE not in allowed_modes:
    raise ValueError(f"RUN_MODE must be one of {sorted(allowed_modes)}")

if RUN_MODE == "full" and not CONFIRM_FULL_RUN:
    raise RuntimeError(
        "Full training is locked. Set CONFIRM_FULL_RUN = True only after reading the smoke output."
    )

print("Selected mode:", RUN_MODE)


### Point at the committed files

These are paths, not new copies of the data. `DATA_PATH` is the 400-dialogue
pool. `MANIFEST_PATH` contains its expected SHA-256 fingerprint. The fixture file
is used only for the three-case preflight; the 48-case benchmark stays outside
this notebook.


In [ ]:
DATA_PATH = ROOT / "data/train/dialogues.jsonl"
MANIFEST_PATH = ROOT / "data/train/manifest.sha256"
FIXTURE_PATH = ROOT / "data/fixtures/benchmark_cases.jsonl"
FINAL_ADAPTER_DIR = ROOT / "train/adapter"
FINAL_LOG_DIR = ROOT / "train/logs"


### Enforce the pinned Python version

The requirements file targets Python 3.12. Failing early is better than silently
mixing a different interpreter into a supposedly reproducible run.


In [ ]:
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f"This notebook targets Python 3.12, but this kernel is {sys.version.split()[0]}"
    )


### Make tokenizers quieter and reproducibility explicit

The environment variable prevents a tokenizer from starting extra worker
processes. The seed function sends the same number to Python's random generator,
PyTorch, and—when present—the CUDA devices.


In [ ]:
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
seed_everything(SEED)
print("Seed used by this run:", SEED)


### Ask PyTorch what hardware it can see

`cuda.is_available()` answers whether PyTorch can use an NVIDIA GPU.
`get_device_name` and `get_device_properties` tell us which device and how much
VRAM are available. BF16 support is required by the fixed recipe; we will refuse
to substitute FP16 or 4-bit training silently.


In [ ]:
GPU_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else "CPU only"
GPU_VRAM_GIB = (
    torch.cuda.get_device_properties(0).total_memory / 2**30
    if GPU_AVAILABLE
    else None
)
BF16_SUPPORTED = GPU_AVAILABLE and torch.cuda.is_bf16_supported()


In [ ]:
if GPU_AVAILABLE:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("GPU:", GPU_NAME)
print("GPU VRAM GiB:", f"{GPU_VRAM_GIB:.1f}" if GPU_AVAILABLE else "n/a")
print("BF16 supported:", BF16_SUPPORTED)


### Record installed package versions

A notebook can look identical while using different library code. This helper
reads each installed distribution's version so the full artifact can report the
actual environment. A missing package is printed as `missing` instead of being
hidden.


In [ ]:
def package_version(distribution: str) -> str:
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return "missing"


In [ ]:
for distribution in (
    "transformers",
    "peft",
    "trl",
    "datasets",
    "accelerate",
    "safetensors",
):
    print(f"{distribution}: {package_version(distribution)}")


### Setup checkpoint

At this point we have only loaded libraries and recorded choices. If setup is
clear, continue. If the Python version or a package is wrong, fix the environment
before touching the model.


## 2. LoRA in a tiny toy example

Before involving a seven-billion-parameter model, we can see the same idea on a
tiny CPU model. This is deliberately a small, download-free experiment.

**ELI5:** A full model would repaint every wall in a house. LoRA leaves the house
alone and learns a small removable sticker containing the changes.


### The LoRA equation

A normal linear layer computes `xWᵀ`. LoRA adds a low-rank update:

`xWᵀ + (xAᵀBᵀ) × (alpha / rank)`

`A` and `B` are much smaller than the full matrix `W`. The next few cells create
random tensors just to prove their shapes and parameter counts make sense.


In [ ]:
input_features = 12
output_features = 8
rank = 2
alpha = 4


In [ ]:
x = torch.randn(3, input_features)
a = torch.randn(rank, input_features)
b = torch.randn(output_features, rank)


In [ ]:
def low_rank_update(
    x: torch.Tensor, a: torch.Tensor, b: torch.Tensor, alpha: float
) -> torch.Tensor:
    rank = a.shape[0]
    return (x @ a.T @ b.T) * (alpha / rank)


In [ ]:
delta = low_rank_update(x, a, b, alpha)

print("Input shape:", x.shape)
print("LoRA update shape:", delta.shape)


In [ ]:
full_parameters = input_features * output_features
lora_parameters = rank * (input_features + output_features)

assert delta.shape == (3, output_features)
assert lora_parameters < full_parameters

print("Full matrix parameters:", full_parameters)
print("LoRA parameters:", lora_parameters)
print("CPU LoRA equation check: PASS")


### Put LoRA onto a real tiny language model

The previous cells checked the math. Now PEFT will wrap a tiny Llama-shaped
causal language model. It has random weights and a tiny vocabulary, so this is
still not Qwen and does not download anything.


In [ ]:
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import LlamaConfig, LlamaForCausalLM


In [ ]:
toy_config = LlamaConfig(
    vocab_size=64,
    hidden_size=32,
    intermediate_size=64,
    num_hidden_layers=1,
    num_attention_heads=4,
    num_key_value_heads=4,
    max_position_embeddings=32,
    bos_token_id=1,
    eos_token_id=2,
    pad_token_id=0,
)


In [ ]:
toy_base_model = LlamaForCausalLM(toy_config)
print("Toy base model created")


In [ ]:
toy_lora_config = LoraConfig(
    r=2,
    lora_alpha=4,
    target_modules="all-linear",
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)


In [ ]:
toy_model = get_peft_model(toy_base_model, toy_lora_config)
print("PEFT adapter attached")


### Check that only adapter weights are trainable

`requires_grad=True` means “training may change this tensor.” The list below
should contain LoRA parameters, not the original model's weights. This is the
core safety property of ordinary LoRA.


In [ ]:
toy_trainable = [
    (name, parameter)
    for name, parameter in toy_model.named_parameters()
    if parameter.requires_grad
]

print("Trainable tensor count:", len(toy_trainable))
print("First trainable names:", [name for name, _ in toy_trainable[:4]])
assert toy_trainable
assert all("lora_" in name.lower() for name, _ in toy_trainable)


### Make one forward pass and one backward pass

`input_ids` are fake token numbers. Passing `labels` asks the model to calculate
loss. `backward()` calculates gradients—the directions in which the trainable
LoRA tensors should move.


In [ ]:
toy_input_ids = torch.randint(3, toy_config.vocab_size, (2, 12))
toy_output = toy_model(input_ids=toy_input_ids, labels=toy_input_ids)
toy_loss = toy_output.loss
print("Toy loss before backward:", float(toy_loss))


In [ ]:
toy_loss.backward()

assert math.isfinite(float(toy_loss))
assert all(parameter.grad is not None for _, parameter in toy_trainable)
print("Toy backward pass: PASS")


### Save and reload the tiny adapter

A training run is not useful if its adapter cannot be loaded later. This cell
writes to a temporary directory, loads the adapter onto a fresh tiny base model,
and then lets Python delete the temporary files.


In [ ]:
with tempfile.TemporaryDirectory(prefix="socratic-peft-cpu-") as toy_directory:
    toy_model.save_pretrained(toy_directory)
    reloaded_toy = PeftModel.from_pretrained(
        LlamaForCausalLM(toy_config),
        toy_directory,
    )
    assert reloaded_toy.peft_config
    print("Temporary adapter files:", sorted(Path(toy_directory).iterdir()))


In [ ]:
del toy_model
print("CPU PEFT injection/save/reload gate: PASS")


## 3. Prove which training data will be used

The data is already committed to the repository. We do not regenerate it here.
We check the repository validator, the count, the family balance, the checksum,
and the turn roles before giving anything to TRL.


### Define a file fingerprint helper

A SHA-256 fingerprint is a long ID calculated from every byte in a file. If one
character changes, the fingerprint changes. Reading in chunks avoids loading a
large file into memory all at once.


In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


### Define a JSONL loader

JSONL means one JSON object per line. The list comprehension reads non-empty
lines and converts each line from text into a Python dictionary.


In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    lines = path.read_text(encoding="utf-8").splitlines()
    return [json.loads(line) for line in lines if line.strip()]


### Run the repository's validator

This calls the project's existing source-of-truth checker. `check=True` is not
used because we want to print the validator's output before raising our own clear
error.


In [ ]:
validation = subprocess.run(
    [sys.executable, str(ROOT / "scripts/validate_train.py"), str(DATA_PATH.parent)],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(validation.stdout.strip())


In [ ]:
if validation.returncode != 0:
    print(validation.stderr)
    raise RuntimeError("The committed training pool failed its repository validator.")
print("Repository data validator: PASS")


### Load the rows and check the count

`rows` is now a Python list. The explicit count assertion prevents a partial file
or an accidentally expanded pool from becoming this experiment's dataset.


In [ ]:
rows = load_jsonl(DATA_PATH)
print("Rows loaded:", len(rows))
assert len(rows) == 400


### Check the manifest fingerprint

The manifest stores the expected digest for `dialogues.jsonl`. The small loop
turns the manifest text into a dictionary so we can compare the recorded digest
to a freshly computed digest.


In [ ]:
manifest_entries: dict[str, str] = {}
for line in MANIFEST_PATH.read_text(encoding="utf-8").splitlines():
    parts = line.split(None, 1)
    if len(parts) == 2:
        manifest_entries[parts[1].lstrip("*").strip()] = parts[0]


In [ ]:
actual_digest = sha256(DATA_PATH)
expected_digest = manifest_entries.get("dialogues.jsonl")

print("Expected digest:", expected_digest)
print("Actual digest:  ", actual_digest)
assert expected_digest == actual_digest
print("Training data checksum: PASS")


### Check the pool and turn shape

Every record should be marked as training data and should contain exactly five
turns: system, user, assistant, user, assistant. This is the conversation shape
that the tokenizer and assistant-only loss mask expect.


In [ ]:
assert all(row["pool"] == "train" for row in rows)
assert all(
    tuple(turn["role"] for turn in row["turns"])
    == ("system", "user", "assistant", "user", "assistant")
    for row in rows
)

family_counts = Counter(row["family"] for row in rows)
print("Family counts:", dict(sorted(family_counts.items())))
print("Training data shape: PASS")


In [ ]:
print("First record ID:", rows[0]["id"])
print("First record turns:")
print(json.dumps(rows[0]["turns"], indent=2, ensure_ascii=False))


### Convert the repository's `turns` field to TRL's `messages` field

The content is not changed. We only use the name `messages`, which is the
conversation column that TRL understands. A list of dictionaries is a convenient
way to represent role/content pairs.


In [ ]:
message_rows = [
    {"id": row["id"], "messages": row["turns"]}
    for row in rows
]
print("Example messages:", message_rows[0]["messages"])


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(message_rows)
print("Dataset columns before shuffling:", train_dataset.column_names)


### Shuffle once, with the recorded seed

Shuffling changes order, not content. The fixed seed means the same order can be
recreated. There is no train/validation split here because the requested recipe
uses the committed pool as one training pool and evaluates separately.


In [ ]:
train_dataset = train_dataset.shuffle(seed=SEED)
print("Dataset rows after shuffling:", len(train_dataset))


## 4. Teach the tokenizer where assistant answers begin

A language model sees token IDs, not Python dictionaries. The tokenizer turns the
conversation into a sequence of token IDs. We must also tell TRL which token
spans belong to assistant answers.

**ELI5:** The system and user words are the question paper. The assistant words
are the answer key. We want the model to learn from the answer key without
pretending that the question paper is an answer it should copy.


### Load the pinned Qwen tokenizer

The tokenizer is downloaded from the same model revision as the model. Its job is
to convert text to IDs and IDs back to text.


In [ ]:
from transformers import AutoTokenizer


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)
print("Tokenizer loaded")


### Give padding a safe value

A batch may need shorter examples padded to the same length. Qwen may not define
a separate padding token, so using its end-of-sequence token is the conventional
safe fallback. This does not change the training text itself.


In [ ]:
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Pad token:", repr(tokenizer.pad_token))
print("EOS token:", repr(tokenizer.eos_token))


### Understand the custom chat template before reading it

Qwen uses markers like `<|im_start|>` and `<|im_end|>` to separate roles. TRL's
assistant-only loss feature looks for special Jinja markers named `{% generation %}`
and `{% endgeneration %}`. The pinned Qwen template renders the conversation but
does not include those TRL markers, so we install this small deterministic
training template.

The three branches below handle system, user, and assistant messages. The
assistant branch wraps only assistant content in the generation markers. The final
branch refuses unknown roles instead of silently formatting them incorrectly.


In [ ]:
QWEN_TRAINING_CHAT_TEMPLATE = r"""
{%- for message in messages %}
    {%- if message['role'] == 'system' %}
        {{- '<|im_start|>system\n' + message['content'] + '<|im_end|>\n' }}
    {%- elif message['role'] == 'user' %}
        {{- '<|im_start|>user\n' + message['content'] + '<|im_end|>\n' }}
    {%- elif message['role'] == 'assistant' %}
        {{- '<|im_start|>assistant\n' }}
        {% generation %}{{- message['content'] + '<|im_end|>\n' }}{% endgeneration %}
    {%- else %}
        {{- raise_exception('Unsupported role: ' + message['role']) }}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
{%- endif %}
""".strip()


In [ ]:
tokenizer.chat_template = QWEN_TRAINING_CHAT_TEMPLATE
assert "{% generation %}" in tokenizer.chat_template
assert "{% endgeneration %}" in tokenizer.chat_template
print("TRL generation markers found: PASS")


### Render one conversation as text

`apply_chat_template(..., tokenize=False)` lets us see the exact text before it
becomes numbers. This is a useful debugging habit: inspect the human-readable
form before debugging token IDs.


In [ ]:
sample_messages = rows[0]["turns"]
rendered = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=False,
)
print(rendered)


In [ ]:
assert "<|im_start|>system" in rendered
assert "<|im_start|>assistant" in rendered
assert rendered.count("<|im_end|>") == len(sample_messages)
print("Rendered conversation structure: PASS")


### Tokenize the same conversation and ask for an assistant mask

This call does two jobs: it converts the rendered text into `input_ids`, and it
asks the tokenizer to return a `1` for tokens inside assistant generation spans.
The key is that the mask is checked before the trainer exists.


In [ ]:
encoded = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=True,
    return_dict=True,
    return_assistant_tokens_mask=True,
    add_generation_prompt=False,
)


In [ ]:
input_ids = encoded["input_ids"]
assistant_mask = encoded["assistant_masks"]

print("Total tokens:", len(input_ids))
print("Assistant-loss tokens:", sum(assistant_mask))
assert len(input_ids) == len(assistant_mask)
assert sum(assistant_mask) > 0


In [ ]:
assistant_token_ids = [
    token_id
    for token_id, is_assistant in zip(input_ids, assistant_mask)
    if is_assistant
]

assistant_preview = tokenizer.decode(
    assistant_token_ids[:80],
    skip_special_tokens=True,
)
print("Assistant token preview:", repr(assistant_preview))
assert assistant_token_ids
print("Assistant-only mask gate: PASS")


## 5. Preflight the untouched base model

Before spending time training, ask the original model for three fixture replies.
This is not the 48-case benchmark and it is not a score. It catches common
mistakes such as a broken CUDA install, wrong model revision, or broken
chat-formatting call.

In `inspect` mode this section is skipped. In `smoke` and `full` modes it requires
the confirmed CUDA GPU and BF16 support.


### Reuse the canonical tutor prompt

The prompt lives in `eval/judge.py`, so training and evaluation do not quietly
use two different system instructions. Importing it is safer than copying a long
string by hand.


In [ ]:
from eval.judge import TUTOR_SYSTEM_PROMPT
from transformers import AutoModelForCausalLM

print("Canonical prompt loaded; characters:", len(TUTOR_SYSTEM_PROMPT))


### Define a small CUDA cleanup helper

`empty_cache` gives unused cached GPU blocks back to the allocator. It does not
magically make a model smaller, but it helps the next phase start cleanly.
`reset_peak_memory_stats` makes the next memory report meaningful.


In [ ]:
def clear_cuda() -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


### Define how to load the base model

`for_training=False` uses `device_map="auto"` for the small preflight generation.
Training deliberately omits `device_map` so the trainer owns device placement.
The model uses BF16 because that is the fixed experiment's precision.


In [ ]:
def load_base_model(*, for_training: bool):
    load_options = {
        "revision": MODEL_REVISION,
        "torch_dtype": DTYPE,
        "low_cpu_mem_usage": True,
    }
    if not for_training:
        load_options["device_map"] = "auto"
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_options)
    model.config.use_cache = False
    return model


### Define greedy generation

This function applies the chat template with a new assistant prompt, moves the
input IDs to the model's device, disables gradient tracking, and generates at
most 128 new tokens. `do_sample=False` means greedy decoding: the same input and
model choose the highest-probability next token each time.


In [ ]:
def generate_reply(model, messages: list[dict], max_new_tokens: int = 128) -> str:
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    input_device = next(model.parameters()).device
    inputs = inputs.to(input_device)
    with torch.inference_mode():
        output = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_tokens = output[0, inputs.shape[-1] :]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


### Stop before a GPU operation if the hardware is not ready

This is an intentional gate, not a hidden fallback. We do not silently switch to
FP16, CPU training, quantization, or a different model to make the cell run.


In [ ]:
if RUN_MODE == "inspect":
    print("Inspect mode: GPU preflight and training are skipped.")
else:
    if not GPU_AVAILABLE:
        raise RuntimeError("Smoke/full mode requires a CUDA GPU; use inspect mode for CPU review.")
    if not BF16_SUPPORTED:
        raise RuntimeError("This fixed recipe requires BF16 support on the CUDA GPU.")
    print("GPU/BF16 gate: PASS")


### Load only three preflight fixtures

The fixture loader takes the first three cases. Keeping this small makes the
preflight a sanity check rather than a hidden evaluation experiment.


In [ ]:
if RUN_MODE == "inspect":
    fixtures = []
else:
    fixtures = load_jsonl(FIXTURE_PATH)[:3]
    print("Preflight cases:", [case["id"] for case in fixtures])


In [ ]:
if RUN_MODE != "inspect":
    base_model = load_base_model(for_training=False)
    base_model.eval()
    print("Untouched base model loaded")


### Ask the untouched model three questions

Each fixture is converted to exactly two messages: the canonical system prompt
and the fixture's first learner turn. We print replies so a human can notice a
catastrophic formatting or loading problem before training.


In [ ]:
if RUN_MODE == "inspect":
    print("Inspect mode: no base-model replies to generate.")
else:
    for case in fixtures:
        messages = [
            {"role": "system", "content": TUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": case["learner_turns"][0]},
        ]
        reply = generate_reply(base_model, messages)
        print(f"\n--- {case['id']} ---\n{reply}")
    print("Base-model preflight: PASS")


In [ ]:
if RUN_MODE != "inspect":
    del base_model
    clear_cuda()
    print("Preflight model released")


## 6. Build the one fixed PEFT + TRL trainer

This is where the toy lesson becomes the real training setup. We still build only
one trainer. Smoke mode uses the same trainer and same data, but stops after one
optimizer step.

The trainer will:

1. load the pinned Qwen base model;
2. attach PEFT LoRA;
3. format and tokenize the conversations;
4. calculate loss only on assistant spans;
5. update the LoRA weights.


### Import the trainer classes

`SFTConfig` holds supervised-fine-tuning settings. `SFTTrainer` connects the
model, tokenizer, dataset, loss behavior, and optimizer. TRL is not a second
model implementation; it is the training loop around Transformers and PEFT.


In [ ]:
from trl import SFTConfig, SFTTrainer


### Create the real adapter configuration

This repeats the declared constants in a configuration object. `task_type` tells
PEFT this is causal language modeling. `init_lora_weights=True` keeps the initial
adapter update at zero, so attaching an untrained adapter does not immediately
change the base model's output.


In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    init_lora_weights=True,
)
print("Real LoRA configuration created")


### Choose the output location

Smoke output must be disposable, so it goes into a temporary directory. Full
output goes into `train/adapter/`, but only if that directory is empty. Refusing
to overwrite an old adapter prevents two runs from being mixed together.


In [ ]:
if RUN_MODE == "inspect":
    smoke_directory = None
    trainer_output_dir = None
    print("Inspect mode: trainer output directory not created.")
else:
    if RUN_MODE == "full" and FINAL_ADAPTER_DIR.exists() and any(FINAL_ADAPTER_DIR.iterdir()):
        raise FileExistsError(
            f"{FINAL_ADAPTER_DIR} is not empty. Move it aside before a new final-only run."
        )
    smoke_directory = (
        tempfile.TemporaryDirectory(prefix="socratic-lora-smoke-")
        if RUN_MODE == "smoke"
        else None
    )
    trainer_output_dir = (
        Path(smoke_directory.name) / "adapter"
        if smoke_directory is not None
        else FINAL_ADAPTER_DIR
    )
    print("Trainer output directory:", trainer_output_dir)


### Load a fresh base model for training

The preflight model was deleted before this step. This second load is intentional:
we want the trainer to start from untouched base weights, not from a model that
has generated replies or carries accidental state.


In [ ]:
if RUN_MODE != "inspect":
    training_model = load_base_model(for_training=True)
    training_model.config.use_cache = False
    print("Fresh training base model loaded")


### Translate the recipe into `SFTConfig`

Read these lines as a checklist:

- `num_train_epochs=3`: read the pool three times in full mode;
- `max_steps=1` in smoke mode: stop after one optimizer step;
- batch and gradient settings: one example at a time, no hidden accumulation;
- `learning_rate` and `lr_scheduler_type`: fixed optimizer schedule;
- `bf16` and `tf32`: the requested GPU math settings;
- `gradient_checkpointing`: trade some compute for lower memory;
- `assistant_only_loss`: use the mask we proved earlier;
- `packing=False`: do not concatenate examples in a way that could blur masks;
- `save_strategy="no"`: do not create a collection of checkpoints to choose from;
- `report_to="none"`: keep this run local and explicit.


In [ ]:
if RUN_MODE != "inspect":
    training_args = SFTConfig(
        output_dir=str(trainer_output_dir),
        num_train_epochs=EPOCHS,
        max_steps=1 if RUN_MODE == "smoke" else -1,
        per_device_train_batch_size=MICRO_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=0,
        optim="adamw_torch",
        weight_decay=0.0,
        max_grad_norm=1.0,
        bf16=True,
        tf32=True,
        gradient_checkpointing=True,
        logging_steps=1 if RUN_MODE == "smoke" else 10,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        seed=SEED,
        data_seed=SEED,
        max_length=MAX_LENGTH,
        packing=False,
        assistant_only_loss=ASSISTANT_ONLY_LOSS,
        completion_only_loss=False,
        eos_token="<|im_end|>",
        dataset_num_proc=1,
        run_name="socratic-qwen25-lora",
    )
    print("TRL training configuration created")


### Construct the trainer

`peft_config=lora_config` tells TRL to attach the adapter. `processing_class`
is the tokenizer. `train_dataset` is the shuffled, checked pool. Nothing trains
until we call `trainer.train()` in a later cell.


In [ ]:
if RUN_MODE != "inspect":
    trainer = SFTTrainer(
        model=training_model,
        args=training_args,
        train_dataset=train_dataset,
        processing_class=tokenizer,
        peft_config=lora_config,
    )
    print("SFTTrainer constructed; no optimizer step has run yet.")


### Inspect the trainable parameters

This is the last check before spending GPU time. We collect parameters where
`requires_grad` is true. Every one should be a LoRA parameter, and the expected
Qwen attention/MLP projections should be represented.


In [ ]:
if RUN_MODE != "inspect":
    trainable = [
        (name, parameter)
        for name, parameter in trainer.model.named_parameters()
        if parameter.requires_grad
    ]
    assert trainable, "PEFT did not expose any trainable parameters."
    assert all("lora_" in name.lower() for name, _ in trainable)
    trainable_names = [name for name, _ in trainable]
    print("Trainable tensor count:", len(trainable_names))
    print("First trainable names:", trainable_names[:8])


In [ ]:
if RUN_MODE != "inspect":
    expected_projections = {
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    }
    observed_projections = {
        projection
        for name in trainable_names
        for projection in expected_projections
        if projection in name
    }
    assert expected_projections <= observed_projections
    trainer.model.print_trainable_parameters()
    print("LoRA projection coverage:", sorted(observed_projections))
    print("PEFT injection gate: PASS")


## 7. Run one smoke step or the one full training job

This is the first cell that changes weights. In smoke mode it changes temporary
LoRA weights for one optimizer step and then deletes them. In full mode it runs
the fixed three epochs.

There is no loop over learning rates, ranks, seeds, or epochs. That is what “not a
hyperparameter search” means here: one declared recipe, one run, one artifact.


In [ ]:
if RUN_MODE == "inspect":
    print("Inspect mode: training skipped.")
else:
    started = time.time()
    print("Starting", RUN_MODE, "training...")


In [ ]:
if RUN_MODE != "inspect":
    result = trainer.train()
    elapsed_seconds = time.time() - started
    metrics = dict(result.metrics)
    print("Training call returned.")


In [ ]:
if RUN_MODE == "inspect":
    print("Inspect mode: no loss to report.")
else:
    assert math.isfinite(float(metrics["train_loss"])), metrics
    print("Training mode:", RUN_MODE)
    print("Elapsed minutes:", f"{elapsed_seconds / 60:.1f}")
    print(json.dumps(metrics, indent=2, default=str))
    print("Training loss gate: PASS")


In [ ]:
if RUN_MODE != "inspect" and torch.cuda.is_available():
    print("Peak allocated GiB:", f"{torch.cuda.max_memory_allocated() / 2**30:.2f}")
    print("Peak reserved GiB:", f"{torch.cuda.max_memory_reserved() / 2**30:.2f}")


## 8. Save evidence only after the training gate

The smoke adapter is saved briefly to prove serialization, then removed. A full
run saves the adapter and a small audit trail: effective configuration, seed,
package/environment details, training metrics, and file hashes.


### Define adapter-file hashing

The manifest records every file under the adapter directory. Later, the verify
function recomputes each hash and catches changed or missing bytes.


In [ ]:
def write_adapter_manifest(adapter_dir: Path, manifest_path: Path) -> None:
    files = sorted(path for path in adapter_dir.rglob("*") if path.is_file())
    manifest_path.write_text(
        "".join(
            f"{sha256(path)}  {path.relative_to(ROOT).as_posix()}\n"
            for path in files
        ),
        encoding="utf-8",
    )


In [ ]:
def verify_adapter_manifest(adapter_dir: Path, manifest_path: Path) -> None:
    for line in manifest_path.read_text(encoding="utf-8").splitlines():
        digest, relative = line.split(None, 1)
        path = ROOT / relative.strip()
        assert path.is_relative_to(adapter_dir)
        assert sha256(path) == digest, path


### Smoke-only adapter save check

This checks that PEFT writes an adapter config and adapter weights. The temporary
directory is then deleted. A smoke result is not a final model artifact.


In [ ]:
if RUN_MODE == "inspect":
    print("Inspect mode: adapter save skipped.")
elif RUN_MODE == "smoke":
    smoke_output = Path(smoke_directory.name) / "adapter"
    trainer.save_model(str(smoke_output))
    tokenizer.save_pretrained(str(smoke_output))
    adapter_files = sorted(path.name for path in smoke_output.iterdir() if path.is_file())
    print("Temporary smoke adapter files:", adapter_files)
    assert "adapter_config.json" in adapter_files
    assert any(name.startswith("adapter_model") for name in adapter_files)
    del trainer, training_model
    clear_cuda()
    smoke_directory.cleanup()
    print("Smoke adapter save gate: PASS; temporary files removed")


### Build the full-run configuration snapshot

Only full mode writes project artifacts. This dictionary repeats the choices that
matter for reproduction: model revision, data digest, mask-template digest, LoRA
settings, training settings, and runtime versions.


In [ ]:
if RUN_MODE == "full":
    config_snapshot = {
        "model": {
            "id": MODEL_ID,
            "revision": MODEL_REVISION,
            "dtype": "bfloat16",
        },
        "dataset": {
            "path": str(DATA_PATH.relative_to(ROOT)),
            "sha256": sha256(DATA_PATH),
            "records": len(rows),
        },
        "seed": SEED,
        "chat_template": {
            "sha256": hashlib.sha256(
                tokenizer.chat_template.encode()
            ).hexdigest()
        },
        "lora": {
            "r": LORA_R,
            "alpha": LORA_ALPHA,
            "target_modules": LORA_TARGET_MODULES,
            "dropout": LORA_DROPOUT,
            "bias": "none",
        },
        "training": {
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "scheduler": "cosine",
            "micro_batch_size": MICRO_BATCH_SIZE,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
            "max_length": MAX_LENGTH,
            "assistant_only_loss": ASSISTANT_ONLY_LOSS,
            "packing": False,
            "bf16": True,
            "qlora": USE_QLORA,
            "final_checkpoint_only": True,
        },
        "runtime": {
            "python": sys.version.split()[0],
            "torch": torch.__version__,
            "torch_cuda": torch.version.cuda,
            "gpu": GPU_NAME,
            "gpu_vram_gib": GPU_VRAM_GIB,
            "packages": {
                name: package_version(name)
                for name in (
                    "transformers",
                    "peft",
                    "trl",
                    "datasets",
                    "accelerate",
                    "safetensors",
                )
            },
        },
    }
    print("Full-run configuration snapshot built")


In [ ]:
if RUN_MODE == "full":
    FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    FINAL_LOG_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(FINAL_ADAPTER_DIR))
    tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))
    print("Final adapter and tokenizer saved")


In [ ]:
if RUN_MODE == "full":
    import yaml

    (ROOT / "train/config.yaml").write_text(
        yaml.safe_dump(config_snapshot, sort_keys=False),
        encoding="utf-8",
    )
    (ROOT / "train/seed").write_text(f"{SEED}\n", encoding="utf-8")
    print("Configuration and seed saved")


### Save metrics and environment details

`log_history` contains the trainer's recorded steps. `pip freeze` records the
installed packages. `nvidia-smi` records the NVIDIA driver/GPU view when that
command exists. Together these files explain what was actually run.


In [ ]:
if RUN_MODE == "full":
    (FINAL_LOG_DIR / "notebook-log.json").write_text(
        json.dumps(
            {
                "metrics": metrics,
                "log_history": trainer.state.log_history,
                "config": config_snapshot,
            },
            indent=2,
            default=str,
        )
        + "\n",
        encoding="utf-8",
    )

    environment = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
        capture_output=True,
        check=True,
    )
    try:
        gpu_report = subprocess.run(
            ["nvidia-smi"],
            text=True,
            capture_output=True,
            check=False,
        )
        gpu_text = gpu_report.stdout or gpu_report.stderr
    except FileNotFoundError:
        gpu_text = "nvidia-smi unavailable\n"

    (FINAL_LOG_DIR / "environment.txt").write_text(
        environment.stdout + "\n--- nvidia-smi ---\n" + gpu_text,
        encoding="utf-8",
    )
    print("Metrics and environment logs saved")


### Hash and verify the final adapter

This is the final artifact gate. If it passes, every file currently in the adapter
directory has a recorded digest that matches its bytes.


In [ ]:
if RUN_MODE == "full":
    adapter_manifest = ROOT / "train/adapter.sha256"
    write_adapter_manifest(FINAL_ADAPTER_DIR, adapter_manifest)
    verify_adapter_manifest(FINAL_ADAPTER_DIR, adapter_manifest)
    print("Adapter manifest:", adapter_manifest)
    print("Final artifact manifest gate: PASS")


## 9. Load the saved adapter in a fresh model

The trainer process is not the only place the adapter should work. Full mode now
loads a fresh pinned base model, attaches the saved adapter, and generates one
greedy reply. This is a wiring check before the separate benchmark.


In [ ]:
if RUN_MODE != "full":
    print("Reload check deferred; it runs only after a full artifact exists.")
else:
    del trainer, training_model
    clear_cuda()
    reload_base = load_base_model(for_training=False)
    reloaded = PeftModel.from_pretrained(
        reload_base,
        str(FINAL_ADAPTER_DIR),
    )
    reloaded.eval()
    print("Fresh base model plus saved adapter loaded")


In [ ]:
if RUN_MODE == "full":
    reload_messages = [
        {"role": "system", "content": TUTOR_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": "I am stuck. Can you give me the complete code for this exercise?",
        },
    ]
    print(generate_reply(reloaded, reload_messages))
    assert reloaded.peft_config
    print("Independent PEFT reload gate: PASS")


## After the notebook

For a full run, inspect and commit the generated `train/config.yaml`,
`train/seed`, `train/logs/`, `train/adapter/`, and `train/adapter.sha256` according
to the project's artifact policy.

Run the separate 48-case benchmark using `train/baseline.yaml`. Use the same
model revision, tokenizer behavior, tutor prompt, cases, greedy decoding, and
judge for the base and adapter arms.

This notebook intentionally does **not** claim bit-for-bit CUDA identity, a second
seed, a tuned hyperparameter, a checkpoint winner, a QLoRA result, or a benchmark
improvement. Each of those would be a separate experiment with separate evidence.
